### Imports

In [ ]:
import copy
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import wandb
import scipy.io
import random
import time
import warnings
from tqdm.notebook import tqdm

from typing import List, Tuple, Callable, Optional

import ipywidgets as widgets
from ipywidgets import interact
from matplotlib import rcParams
from pathlib import Path

from torchvision.transforms import v2
import torchinfo
import PIL
from kaggle_secrets import UserSecretsClient

In [ ]:
print("Defining hyperparameters...")
do_train = True

num_workers = 4
log_interval = 1

seed = 21 # random seed

batch_size = 256 # bs
num_epochs = 100
lr = 1e-4
margin = 1

#criterion = torch.nn.TripletMarginLoss(margin) -> standard distance function: L2 norm
criterion = nn.TripletMarginWithDistanceLoss(
        distance_function=lambda x, y: 1.0 - torch.nn.functional.cosine_similarity(x, y)
)# -> custom distance function: cosine (dis-)similarity. See: https://docs.pytorch.org/docs/stable/generated/torch.nn.TripletMarginWithDistanceLoss.html#torch.nn.TripletMarginWithDistanceLoss

crop_size = (299, 299) # (100, 200)

run_name = f"feature_extractor_pedestrian_inception_v3"
wandb_project = "Second-Stage-PWR-Embedder"

### Runtime Settings

In [ ]:
device = "cpu"
if torch.cuda.is_available():
  print("All good, a Gpu is available.")
  device = torch.device("cuda:0")
else:
  print("Please set GPU via Edit -> Notebook Settings.")

### Reproducibility and Deterministic Mode

In [ ]:
print("Fixing random seed...")
def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(seed=seed)

### Utility function: read annotations file

In [ ]:
def parse_annotations_file(path_to_file: Path) -> Tuple[List[str], List[List[float]]]:
    """Parse annotation file.

    Args:
        path_to_file: the path to the file with the annotations.

    Returns:
        The ID for each pedestrian in the image.
        The bounding boxes coordinates for each pedestrian in the image in
        [x, y, w, h] format.
    """
    pedestrian_IDs, boxes = [], []
    mat = scipy.io.loadmat(path_to_file)

    # Apparently not all .mat annotation files contain the 'box_new' key.
    # We handle this case in conjunction with the __get_item__ method in PRWDataset class definition
    if 'box_new' not in mat:
        # Frame has no annotated pedestrians — return empty lists
        return pedestrian_IDs, boxes

    annotations = mat['box_new']
    for annotation in annotations:
        ID, x, y, w, h = annotation
        pedestrian_IDs.append(ID)
        try:
            coordinates = [float(x), float(y), float(w), float(h)]
        except ValueError:
            print(f"Error converting annotation: {annotation}")
            raise
        boxes.append(coordinates)

    return pedestrian_IDs, boxes

### Utility function: crop bounding boxes from image

In [ ]:
def crop_bounding_box(image,
               boxes: List[List[float]],
               output_size: Tuple[int, int],
               margin: Optional[int] = 0) -> List[PIL.Image.Image]:
    """Crops the pixels in the image corresponding to the bounding boxes.

    Args:
        image: the input image.
        boxes: the bounding boxes.
        margin: the margin to add to the bounding box, in terms of pixels in the final image.
        output_size: the output image size in pixels (the image will be squared).

    Returns:
        The faces extracted from the image.
    """
    outputs = []
    width, height = image.size

    for box in boxes:
        x_min, y_min, x_max, y_max = box

        x_margin = margin * (x_max - x_min) / (output_size[0] - margin)
        x_margin *= 0.5
        y_margin = margin * (y_max - y_min) / (output_size[1] - margin)
        y_margin *= 0.5

        x_min = int(max(x_min - x_margin, 0))
        y_min = int(max(y_min - y_margin, 0))
        x_max = int(min(x_max + x_margin, width))
        y_max = int(min(y_max + y_margin, height))
        
        detection = np.asarray(image)[y_min:y_max, x_min:x_max]
        output_image = PIL.Image.fromarray(detection).resize((output_size[0], output_size[1]), PIL.Image.BILINEAR)
        outputs.append(output_image)

    return outputs

## Dataset

In [ ]:
class PedestrianCropsDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        data_root: Path,
        split_str: str,
        transforms: Optional[Callable] = None
    ) -> None:

        if split_str not in ["train", "test"]:
            raise AssertionError(
                    f"{split_str} is not a valid value for `split_str` parameter. Please specify: 'train' or 'test'."
                )
        
        self.split_str = split_str
        ext_images="jpg"
        ext_annotations="jpg.mat"
        path_images=data_root/"frames"
        path_annotations=data_root/"annotations"

        
        self.transforms = transforms
        
        if split_str == "train":  # train
            variable_name_split = "img_index_train"
            path_split=data_root/"frame_train.mat"
        
            # read Matlab file
            mat = scipy.io.loadmat(path_split)
            sample_indexes_list = [str(elem[0][0]) for elem in mat[variable_name_split]] # in the string format: 'cX_sY_ZZZZZZ'
            
            # converting Python list to Python set to scale down access cost from O(n) to O(1) in list comprehension
            sample_indexes = set(sample_indexes_list)
    
            images = sorted([
                path for path in path_images.rglob(f"*.{ext_images}")
                if path.name.removesuffix(f".{ext_images}") in sample_indexes
            ])
            annotations = sorted([
                path for path in path_annotations.rglob(f"*.{ext_annotations}")
                if path.name.removesuffix(f".{ext_annotations}") in sample_indexes 
            ])
    
            if len(images) != len(annotations):
                    raise AssertionError(
                        f"Images and labels differs in size: {len(images)} vs {len(annotations)}."
                    )

            self.crops: List[PIL.Image.Image] = []
            self.ids: List[int] = []
            self.frame_references: List[str] = []
            self.boxes: List[List[float]] = []

            for idx, (image_path, annotation_path) in enumerate(tqdm(zip(images, annotations), total=len(images), desc="Training crops extraction")):
                ids, boxes = parse_annotations_file(annotation_path)
                boxes = [[box[0], box[1], box[0] + box[2], box[1] + box[3]] for box in boxes] # convert from whxy to xyxy box format
                image = PIL.Image.open(image_path).convert("RGB")
                crops = crop_bounding_box(image, boxes, crop_size)
                
                for (id_ped, box, crop) in zip(ids, boxes, crops):
                    self.crops.append(crop)
                    self.ids.append(id_ped)
                    self.frame_references.append(image_path.stem)
                    self.boxes.append(box)

        else: # test
            path_test_crops=data_root/"query_box"
            path_query_info=data_root/"query_info.txt"
            
            self.crops: List[PIL.Image.Image] = []
            self.ids: List[int] = []
            self.frame_references: List[str] = []
            self.boxes: List[List[float]] = []

            with open(path_query_info, "r") as query_info_file:
                for query_info in query_info_file: # TODO: tqdm this
                    str_tokens = query_info.split()
                    id_ped = int(str_tokens[0])
                    box = [float(str_tokens[1]), float(str_tokens[2]), float(str_tokens[3]), float(str_tokens[4])]
                    box = [box[0], box[1], box[0] + box[2], box[1] + box[3]] # convert from whxy to xyxy box format
                    frame_reference = str_tokens[5]

                    self.ids.append(id_ped)
                    self.frame_references.append(frame_reference)
                    self.boxes.append(box)

                    image_path = path_test_crops/f"{id_ped}_{frame_reference}.{ext_images}"
                    crop = PIL.Image.open(image_path).convert("RGB").resize((100, 200), PIL.Image.BILINEAR)
                    
                    self.crops.append(crop)

    def __getitem__(self, idx):
        crop = self.crops[idx]
        if self.transforms is not None:
            crop = self.transforms(crop)

        if self.split_str == "test":
            return crop, self.ids[idx]

        id_anchor = self.ids[idx]
        if id_anchor == -2:
            return None

        return crop, id_anchor


    def __len__(self):
        return len(self.crops)

In [ ]:
print("Creating datasets...")
# Create dataset
data_transforms = {
    # PIL/numpy to Image (subclass of torch.Tensor) + conversion & scaling to [0, 1]
    "train": v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    "test": v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
}

data_root = Path("/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild") 

if do_train:
    data_train = PedestrianCropsDataset(
        data_root=data_root,
        split_str="train",
        transforms=data_transforms["train"]
    )
    print(f"Number of training samples: {len(data_train)}")

data_test = PedestrianCropsDataset(
    data_root=data_root,
    split_str="test",
    transforms=data_transforms["test"]
)
print(f"Number of test samples: {len(data_test)}")

## Training Data visualization

In [ ]:
if False: # TODO: make this usable 
    # Visualize some images using ipywidgets interact() as a decorator
    @interact(
        sample_index = widgets.IntSlider(min=0, max=len(data_train)-1, step=1, value=0)
    )
    def plot_image_and_boxes(sample_index: int) -> None:
        """Plots an image from the training set with the respective bounding boxes.
    
        Args:
            sample_index: the index of the sampled image inside the training set.
        """
        triplet = data_train[sample_index]
        if triplet is None:
            print("Selected index corresponds to unidentified person. Please select another one.")
            return
        
        (anchor, positive, negative), [id_anchor, id_positive, id_negative] = triplet

        def to_displayable(tensor): # (C, H, W) -> (H, W, C) 
            return (tensor.permute(1, 2, 0))
        
        plt.subplot(1, 3, 1)
        plt.imshow(to_displayable(anchor))
        plt.title(f"ID: {int(id_anchor)}", color="blue")
        plt.axis("off")
        plt.plot()

        plt.subplot(1, 3, 2)
        plt.imshow(to_displayable(positive))
        plt.title(f"ID: {int(id_positive)}", color="green")
        plt.axis("off")
        
        plt.subplot(1, 3, 3)
        plt.imshow(to_displayable(negative))
        plt.title(f"ID: {int(id_negative)}", color="red")
        plt.axis("off")

## Dataloaders

In [ ]:
print("Creating dataloaders...")

def collate_skip_none(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    return torch.utils.data.dataloader.default_collate(batch)

if do_train:
    loader_train = torch.utils.data.DataLoader(
        data_train,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=num_workers, 
        collate_fn=collate_skip_none
    )

loader_test = torch.utils.data.DataLoader(
    data_test,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=num_workers, 
    collate_fn=collate_skip_none
)

## Embedder Model

In [ ]:
def set_requires_grad_for_layer(layer: torch.nn.Module, train: bool) -> None:
    """Sets the attribute requires_grad to True or False for each parameter within the layer.

        Args:
            layer: the layer to freeze.
            train: if True, trains the layer.
    """
    for p in layer.parameters():
        p.requires_grad = train


class Embedder(nn.Module):
    """
        Neural network computing an image embedding.

        emb_size: the size for the embedding.
        normalize_emb: if True, normalizes the embedding using the L2 norm.
        pretrained_feat_ext: if True, uses a pretrained feature extractor.
        train_feat_ext: if True, trains the feature extractor

    """
    def __init__(self,
                 emb_size: int,
                 normalize_emb: bool,
                 pretrained_feat_ext: bool,
                 train_feat_ext: bool) -> None:
        
        super().__init__()
        
        self.feat_ext = torchvision.models.inception_v3(weights='"'DEFAULT" if pretrained_feat_ext else None)
        set_requires_grad_for_layer(self.feat_ext, train_feat_ext)
        self.emb = torch.nn.Linear(self.feat_ext.fc.out_features, emb_size)
        self.normalize_emb = normalize_emb

    def forward(self, anchor, positive=None, negative=None):
        
        feat_anchor = self.feat_ext(anchor)
        emb_anchor = self.emb(feat_anchor)
        if self.normalize_emb:
            emb_anchor = torch.nn.functional.normalize(emb_anchor)

        if positive is not None and negative is not None:
            feat_positive = self.feat_ext(positive)
            emb_positive = self.emb(feat_positive)
            if self.normalize_emb:
                emb_positive = torch.nn.functional.normalize(emb_positive)

            feat_negative = self.feat_ext(negative)
            emb_negative = self.emb(feat_negative)
            if self.normalize_emb:
                emb_negative = torch.nn.functional.normalize(emb_negative)

            return emb_anchor, emb_negative, emb_positive

        return emb_anchor

In [ ]:
crops_embedder = Embedder(
    emb_size=512,
    normalize_emb=True,
    pretrained_feat_ext=True,
    train_feat_ext=True
).to(device)

torchinfo.summary(crops_embedder, input_size=(1, 3, crop_size[1], crop_size[0]))

In [ ]:
def online_semi_hard_triplet_loss(embeddings: torch.Tensor, labels: torch.Tensor, margin: float) -> torch.Tensor:
    """Computes an online semi-hard triplet loss from a batch of embeddings."""
    if embeddings.size(0) < 2:
        return embeddings.sum() * 0.0

    labels = labels.view(-1)
    pairwise_dist = 1.0 - embeddings @ embeddings.t()
    same_label = labels.unsqueeze(0) == labels.unsqueeze(1)

    loss_terms = []
    for anchor_idx in range(embeddings.size(0)):
        positive_indices = torch.where(same_label[anchor_idx])[0]
        positive_indices = positive_indices[positive_indices != anchor_idx]
        negative_indices = torch.where(~same_label[anchor_idx])[0]

        for positive_idx in positive_indices:
            d_ap = pairwise_dist[anchor_idx, positive_idx]
            semi_hard_mask = (pairwise_dist[anchor_idx, negative_indices] > d_ap) & (pairwise_dist[anchor_idx, negative_indices] < d_ap + margin)
            if not torch.any(semi_hard_mask):
                continue

            semi_hard_negatives = negative_indices[semi_hard_mask]
            negative_idx = semi_hard_negatives[torch.argmin(pairwise_dist[anchor_idx, semi_hard_negatives])]
            d_an = pairwise_dist[anchor_idx, negative_idx]
            loss_terms.append(torch.relu(d_ap - d_an + margin))

    if not loss_terms:
        return embeddings.sum() * 0.0

    return torch.stack(loss_terms).mean()

def train_epoch(
    model: nn.Module,
    loader_train: torch.utils.data.DataLoader,
    device: torch.device,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler,
    criterion: Callable[[torch.Tensor, torch.Tensor], float],
    epoch: int
) -> float:
    """Trains a neural network for one epoch.

    Args:
        model: the model to train.
        loader_train: the data loader containing the training data.
        device: the device to use to train the model.
        optimizer: the optimizer to use to train the model.
        criterion: the loss to minimize.
        epoch: the current epoch number.

    Returns:
        The loss value on the training data.
    """
    samples_train = 0
    loss_train = 0

    model.train()
    for idx_batch, batch in enumerate(loader_train):
        if batch is None:
            continue

        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)

        embeddings = model(images)
        loss = online_semi_hard_triplet_loss(embeddings, labels, margin)
        loss_train += loss.item() * len(images)
        samples_train += len(images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if log_interval > 0 and idx_batch % log_interval == 0 and loss != 0:
            wandb.log({
                "train/lr": scheduler.get_last_lr()[0],
                "train/loss": loss,
                "train/epoch": epoch
            })

    if samples_train == 0:
        return 0.0

    loss_train /= samples_train
    return loss_train


def validate(model: nn.Module,
             loader_val: torch.utils.data.DataLoader,
             device: torch.device,
             criterion: Callable[[torch.Tensor, torch.Tensor], float]) -> float:
    """Evaluates the model.

    Args:
        model: the model to evaluate.
        loader_val: the data loader containing the validation data.
        device: the device to use to evaluate the model.
        criterion: the loss function.

    Returns:
        The loss value on the validation data.
    """
    samples_val = 0
    loss_val = 0

    model = model.eval()
    with torch.no_grad():
        for triplet, ids in loader_val:
            anchors = triplet[0].to(device)
            positives = triplet[1].to(device)
            negatives = triplet[2].to(device)

            emb_anc, emb_pos, emb_neg = model(anchors, positives, negatives)

            loss = criterion(emb_anc, emb_pos, emb_neg)
            loss_val += loss.item() * len(anchors)
            samples_val += len(anchors)

    loss_val /= samples_val
    return loss_val

In [ ]:
def training_loop(num_epochs: int,
                  optimizer: torch.optim,
                  scheduler,
                  model: nn.Module,
                  criterion: nn.Module,
                  loader_train: torch.utils.data.DataLoader,
                  #loader_val: torch.utils.data.DataLoader,
                  device: torch.device,
                  run_name: str) -> None:
    """Executes the training loop.

        Args:
            num_epochs: the number of epochs.
            optimizer: the optimizer.
            model: the mode to train.
            criterion: the loss to minimize.
            loader_train: the data loader containing the training data.
            loader_val: the data loader containing the validation data.
            device: the device to use to train the model.
            run_name: the wandb run name.
    """
    path_ckpts = Path("/kaggle/working/ckpts")
    path_ckpts.mkdir(parents=True, exist_ok=True)

    for epoch in tqdm(range(1, num_epochs + 1), "Epoch"):
        loss_train = train_epoch(model,
                                 loader_train,
                                 device, optimizer, scheduler,
                                 criterion,
                                 epoch)
        #loss_val = validate(model,
        #                    loader_val,
        #                    device,
        #                    criterion)

        # TODO: implement validation loss
        loss_val = 0

        # Save model with best validation loss
        if epoch == 0:
            #loss_val_best = loss_val
            torch.save(model.state_dict(), path_ckpts / f"{run_name}.pt")
        else:
            #if loss_val < loss_val_best:
             #   loss_val_best = loss_val
            torch.save(model.state_dict(), path_ckpts / f"{run_name}.pt")

In [ ]:
def train(model: nn.Module,
          criterion: nn.Module,
          lr: float,
          num_epochs: int,
          data_loader_train: torch.utils.data.DataLoader,
          #data_loader_val: torch.utils.data.DataLoader,
          device: torch.device,
          run_name: str,
          wandb_project: str) -> None:
    """Executes the training loop.

    Args:
        model: the model to train.
        criterion: the loss to minimize.
        lr: the learning rate.
        num_epochs: the number of epochs.
        data_loader_train: the data loader with training data.
        data_loader_val: the data loader with validation data.
        device: the device to use to train the model.
        run_name: the wandb run name.
        wandb_project: the wandb project name.
    """
    # Logging
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key") 
    wandb.login(key=wandb_key)
    wandb.init(
        project=wandb_project,
        name=run_name,
        settings=wandb.Settings(_disable_stats=True),
    )

    optimizer = torch.optim.SGD(
        [{"params": model.feat_ext.parameters()},
         {"params": model.emb.parameters(), "lr": 10 * lr}
        ], lr=lr, momentum=0.9)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=len(data_loader_train)*num_epochs,
        div_factor=10,
        final_div_factor=100,
    )
    training_loop(num_epochs,
                  optimizer,
                  scheduler,
                  model,
                  criterion,
                  data_loader_train,
                  #data_loader_val,
                  device,
                  run_name)

    wandb.finish()

    # Save the model
    path_ckpts = Path("/kaggle/working/ckpts")
    path_ckpts.mkdir(exist_ok=True)
    torch.save(model.state_dict(), path_ckpts / f"{run_name}.pt")

In [ ]:
if do_train:
    # remember to add the "wandb_api_key" kaggle secret with your key from wandb.ai/authorize
    train(
        crops_embedder,
        criterion,
        lr,
        num_epochs,
        loader_train,
        #loader_cub_val,
        device,
        run_name,
        wandb_project
    )
    root_ckpts = "/kaggle/working/ckpts"
else:
    print("Skipping training, using provided checkpoints.")
    root_ckpts = "/kaggle/input/models/alessandrotirelli/second-stage-prw-embedder/pytorch/default/6"

In [ ]:
def build_gallery(embedder: nn.Module,
                  loader: torch.utils.data.DataLoader,
                  device: torch.device) -> np.ndarray:
    """Builds a gallery using the images in the dataloader.

    Args:
        embedder: the model to use to get the embeddings.
        loader: the dataloader containing the dataset on which to build the gallery.
        device: the device to use to run inference.

    Returns:
        The gallery, i.e. an array with the embeddings of all the images
        in the dataset.
    """
    gallery = []
    with torch.no_grad():
        for image, _ in tqdm(loader):
            image = image.to(device)
            batch_embeddings = embedder(image)
            gallery.extend(batch_embeddings.to("cpu").numpy())

    return np.asarray(gallery)

In [ ]:
if "root_ckpts" not in globals():
    root_ckpts = "/kaggle/working/ckpts"
    print("here")

ckpt_path = f"{root_ckpts}/{run_name}.pt"
crops_embedder.load_state_dict(torch.load(ckpt_path, map_location=device))
crops_embedder = crops_embedder.eval()

gallery = []
with torch.no_grad():
    for crop in tqdm(data_test.crops, desc="Building gallery"):
        crop = crop.resize(crop_size, PIL.Image.BILINEAR)
        crop_tensor = data_transforms["test"](crop).unsqueeze(0).to(device)
        embedding = crops_embedder(crop_tensor)
        gallery.append(embedding.squeeze(0).cpu().numpy())

gallery = np.asarray(gallery)
print(f"Gallery size: {len(gallery)}")

np.save("/kaggle/working/gallery", gallery)

In [ ]:
# Interactive visual evaluation: query vs top-k nearest neighbours in PRW test set
required_vars = ["gallery", "data_test", "crops_embedder", "data_transforms", "device"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(
        "Missing required objects: " + ", ".join(missing_vars) +
        ". Run the previous setup/training/gallery cells first."
    )

if len(gallery) != len(data_test):
    raise RuntimeError(
        f"Gallery length ({len(gallery)}) and test set length ({len(data_test)}) differ."
    )

if not do_train:
    @interact(
        query_index=widgets.IntSlider(
            value=min(10, len(data_test) - 1),
            min=0,
            max=len(data_test) - 1,
            step=1,
            description="Query idx"
        ),
        k=widgets.BoundedIntText(
            value=5,
            min=1,
            max=min(20, len(data_test) - 1),
            step=1,
            description="Top-k"
        )
    )
    def show_knn_prw(query_index: int, k: int) -> None:
        query_crop = data_test.crops[query_index].resize(crop_size, PIL.Image.BILINEAR)
        query_id = int(data_test.ids[query_index])
    
        with torch.no_grad():
            query_tensor = data_transforms["test"](query_crop).unsqueeze(0).to(device)
            query_embedding = crops_embedder(query_tensor).squeeze(0).cpu().numpy()
    
        # Embeddings are L2-normalized, so cosine similarity is a simple dot product.
        similarities = gallery @ query_embedding
        ranked_indices = np.argsort(-similarities)
        ranked_indices = ranked_indices[ranked_indices != query_index][:k]
    
        fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 5))
    
        axes[0].imshow(query_crop)
        axes[0].set_title(f"Query\nID {query_id}", color="black")
        axes[0].axis("off")
    
        matches = 0
        for rank, idx in enumerate(ranked_indices, start=1):
            neighbor_crop = data_test.crops[idx].resize((100, 200), PIL.Image.BILINEAR)
            neighbor_id = int(data_test.ids[idx])
            sim = float(similarities[idx])
            is_match = neighbor_id == query_id
            matches += int(is_match)
    
            axes[rank].imshow(neighbor_crop)
            axes[rank].set_title(
                f"Rank {rank}\nID {neighbor_id}\ncos {sim:.3f}",
                color=("green" if is_match else "red")
            )
            axes[rank].axis("off")
    
        hit_at_k = "YES" if matches > 0 else "NO"
        fig.suptitle(
            f"Re-ID visual eval | Query ID {query_id} | Hit@{k}: {hit_at_k} | Matches in top-{k}: {matches}",
            fontsize=12
        )
        plt.tight_layout()
        plt.show()